# 사람 metric depth 파이프라인 (Sapiens 백본)
## SAM2 → crop → Sapiens depth → 사람 LiDAR anchor → (normal refine) → 키/너비 검증

**백본 = Sapiens depth 단일 고정.** DA v2 아님 (DA v2 는 별도 *씬* 브랜치이고 로컬에서 따로 돌림).
- Sapiens 는 torchscript 입력 **1024×768 고정** → crop 을 3:4 로만 맞추면 되고, **patch 배수(14/16) 제약 없음**.
- fit 형태는 자동판별: Sapiens(깊이) 면 `z=a·s+b`, 만약 disparity 면 `1/z=a·s+b` 가 자동 선택됨.

```
Sapiens Seg 1B 전신 마스크
→ 3:4 crop
→ Sapiens Depth 1B
→ 사람 LiDAR anchor
→ Normal 1B refine(optional)
→ 키/너비 검증
```

anchor 는 **사람 LiDAR 만**(conf==2, 마스크 erode) → 씬 foreground-bias 차단.
실행 전: 런타임 → GPU. 라이선스 Sapiens CC BY-NC 4.0 / SAM2 Apache-2.0.


In [ ]:
# 1. 환경
import torch
print("CUDA:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
print("torch:", torch.__version__)


- SAM2 설치 안 함
- Seg / Depth / Normal 모델 파일이 정상 크기로 이미 있으면 재다운로드 안 함
- 파일이 너무 작으면 다운로드 실패 파일로 보고 삭제 후 다시 받음

In [ ]:
# 2. SAM2 설치 (Sapiens torchscript 는 의존성 없음)
# !pip -q install 'git+https://github.com/facebookresearch/sam2.git' 2>/dev/null
# import importlib
# print("sam2 OK:", importlib.util.find_spec("sam2") is not None)


## 3. 입력
프레임당 **RGB + LiDAR depth(meter) + confidence(0/1/2) + intrinsic(RGB)**.
- RGB 만으론 anchor 불가(scale 못 박음). depth/conf 필수, **키 검증엔 intrinsic 필수**.
- depth/conf 해상도 달라도 자동으로 RGB 해상도에 맞춤(nearest).
- intrinsics.json
= RGB 원본 이미지 해상도 기준 fx / fy / cx / cy
```
inputs/  KEY_rgb.png  KEY_depth.npy(m)  KEY_conf.npy(0/1/2)  ...  intrinsics.json
intrinsics.json = {"KEY":{fx,fy,cx,cy}, ...}  또는 {"_default":{...}}
```


In [ ]:
# 3a. 업로드 (inputs.zip 권장)
import os, zipfile
from google.colab import files
os.makedirs("inputs", exist_ok=True)
up = files.upload()
for name in up:
    if name.endswith(".zip"):
        zipfile.ZipFile(name).extractall("inputs"); print("unzip:", name)
    else:
        os.replace(name, f"inputs/{name}")
print("inputs/:", sorted(os.listdir("inputs"))[:20])


In [ ]:
# 3b. 프레임 목록 (파일명 POSE_view_fid_rgb.png 자동 파싱)
import glob, os, json, re
FRAMES=[]
for p in sorted(glob.glob("inputs/*_rgb.png"))+sorted(glob.glob("inputs/*_rgb.jpg")):
    key=os.path.basename(p).rsplit("_rgb.",1)[0]
    m=re.match(r"([A-Za-z0-9]+)_([a-z]+)_(\w+)", key)
    pose,view,fid=(m.group(1),m.group(2),m.group(3)) if m else (key,key,key)
    d=f"inputs/{key}_depth.npy"; cf=f"inputs/{key}_conf.npy"
    FRAMES.append(dict(key=key,pose=pose,view=view,fid=fid,rgb=p,
                       depth=d if os.path.exists(d) else None, conf=cf if os.path.exists(cf) else None))
INTR=json.load(open("inputs/intrinsics.json")) if os.path.exists("inputs/intrinsics.json") else {}
def get_intr(key,W,H):
    d=INTR.get(key) or INTR.get("_default")
    if d: return d['fx'],d['fy'],d['cx'],d['cy']
    f=1.0*max(W,H); return f,f,W/2,H/2   # 근사(키 정확도 위해 실제 intrinsic 권장)
SUBJECT_HEIGHT_M=None   # 실제 키 입력 시 오차 비교
for f in FRAMES: print(f["key"],"| depth",bool(f["depth"]),"| conf",bool(f["conf"]))
print("총",len(FRAMES),"프레임")


In [ ]:
# 4. 공용 함수
import os
import subprocess
import numpy as np
import cv2
import torch


def ensure_model(url, path, min_gb=3.0):
    """
    정상 크기의 모델 파일이 존재하면 다운로드를 생략.
    작은 파일은 다운로드 실패 파일로 보고 삭제 후 다시 받음.
    """

    min_bytes = int(min_gb * 1024**3)

    if os.path.exists(path):
        size = os.path.getsize(path)

        if size >= min_bytes:
            print(
                f"✓ 모델 존재 → 다운로드 생략\n"
                f"  {path} ({size / 1024**3:.2f} GB)"
            )
            return

        print(
            f"⚠ 불완전 모델 파일 삭제\n"
            f"  {path} ({size / 1024**2:.1f} MB)"
        )
        os.remove(path)

    temp_path = path + ".part"

    if os.path.exists(temp_path):
        os.remove(temp_path)

    print("다운로드 시작:", os.path.basename(path))

    subprocess.run(
        [
            "wget",
            "-q",
            "--show-progress",
            "--tries=3",
            "--timeout=30",
            "-O",
            temp_path,
            url,
        ],
        check=True,
    )

    downloaded_size = (
        os.path.getsize(temp_path)
        if os.path.exists(temp_path)
        else 0
    )

    if downloaded_size < min_bytes:
        if os.path.exists(temp_path):
            os.remove(temp_path)

        raise RuntimeError(
            f"모델 다운로드 실패 또는 불완전 파일\n"
            f"받은 용량: {downloaded_size / 1024**2:.1f} MB\n"
            f"URL: {url}"
        )

    os.replace(temp_path, path)

    print(
        f"✓ 다운로드 완료\n"
        f"  {path} ({downloaded_size / 1024**3:.2f} GB)"
    )


def load_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)

    if img is None:
        raise FileNotFoundError(path)

    # RGBA PNG가 있으면 alpha를 전신 마스크로 우선 사용
    if img.ndim == 3 and img.shape[2] == 4:
        rgb = cv2.cvtColor(img[:, :, :3], cv2.COLOR_BGR2RGB)
        alpha = img[:, :, 3] > 128
        return rgb, alpha

    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB), None


def to_rgb_res(a, W, H, nearest=True):
    if a is None or a.shape[:2] == (H, W):
        return a

    return cv2.resize(
        a,
        (W, H),
        interpolation=(
            cv2.INTER_NEAREST
            if nearest
            else cv2.INTER_LINEAR
        ),
    )


def pad_bbox_3_4(x0, y0, x1, y1, margin=0.18, ratio=0.75):
    bw = max(1, x1 - x0)
    bh = max(1, y1 - y0)

    cx = (x0 + x1) / 2
    cy = (y0 + y1) / 2

    bw *= 1 + margin
    bh *= 1 + margin

    if bw / bh > ratio:
        bh = bw / ratio
    else:
        bw = bh * ratio

    return (
        cx - bw / 2,
        cy - bh / 2,
        cx + bw / 2,
        cy + bh / 2,
    )


def crop_pad(arr, box, fill=0):
    x0, y0, x1, y1 = [int(round(v)) for v in box]

    cw = max(1, x1 - x0)
    ch = max(1, y1 - y0)

    if arr.ndim == 3:
        out = np.full(
            (ch, cw, arr.shape[2]),
            fill,
            arr.dtype,
        )
    else:
        out = np.full(
            (ch, cw),
            fill,
            arr.dtype,
        )

    H, W = arr.shape[:2]

    sx0 = max(0, x0)
    sy0 = max(0, y0)
    sx1 = min(W, x1)
    sy1 = min(H, y1)

    if sx1 > sx0 and sy1 > sy0:
        dx = sx0 - x0
        dy = sy0 - y0

        out[
            dy:dy + (sy1 - sy0),
            dx:dx + (sx1 - sx0),
        ] = arr[sy0:sy1, sx0:sx1]

    return out, (x0, y0, x1, y1)


def ransac_lin(x, y, iters=3000, k=3.0, seed=0):
    rng = np.random.default_rng(seed)

    best_inlier = None
    best_count = -1
    n = len(x)

    for _ in range(iters):
        i, j = rng.integers(0, n, 2)

        dx = x[i] - x[j]

        if abs(dx) < 1e-9:
            continue

        a = (y[i] - y[j]) / dx
        b = y[i] - a * x[i]

        residual = np.abs(y - (a * x + b))
        mad = (
            np.median(
                np.abs(
                    residual - np.median(residual)
                )
            )
            * 1.4826
            + 1e-12
        )

        inlier = residual < k * mad
        count = int(inlier.sum())

        if count > best_count:
            best_count = count
            best_inlier = inlier

    if best_inlier is None or best_inlier.sum() < 2:
        raise RuntimeError("RANSAC 선형 적합 실패")

    for _ in range(2):
        A = np.stack(
            [
                x[best_inlier],
                np.ones(int(best_inlier.sum())),
            ],
            axis=1,
        )

        a, b = np.linalg.lstsq(
            A,
            y[best_inlier],
            rcond=None,
        )[0]

        residual = np.abs(y - (a * x + b))
        mad = (
            np.median(
                np.abs(
                    residual - np.median(residual)
                )
            )
            * 1.4826
            + 1e-12
        )

        best_inlier = residual < k * mad

    return float(a), float(b), best_inlier


def anchor_auto(s, z):
    aA, bA, iA = ransac_lin(s, z)

    relA = np.median(
        np.abs((aA * s + bA) - z)
        / np.clip(z, 1e-6, None)
    )

    aB, bB, iB = ransac_lin(
        s,
        1.0 / np.clip(z, 1e-6, None),
    )

    zB = 1.0 / np.clip(
        aB * s + bB,
        1e-6,
        None,
    )

    relB = np.median(
        np.abs(zB - z)
        / np.clip(z, 1e-6, None)
    )

    if relA <= relB:
        return dict(
            mode="depth",
            a=float(aA),
            b=float(bA),
            rel=float(relA),
            inlier=float(iA.mean()),
        )

    return dict(
        mode="disparity",
        a=float(aB),
        b=float(bB),
        rel=float(relB),
        inlier=float(iB.mean()),
    )


def apply_anchor(s, p):
    if p["mode"] == "depth":
        return p["a"] * s + p["b"]

    return 1.0 / np.clip(
        p["a"] * s + p["b"],
        1e-6,
        None,
    )


def measure_height_m(z, mask, fx, fy, cx, cy):
    v, u = np.where(
        mask
        & (z > 0)
        & np.isfinite(z)
    )

    if len(v) < 50:
        return None

    Y = (v - cy) * z[v, u] / fy

    return float(
        np.percentile(Y, 99.5)
        - np.percentile(Y, 0.5)
    )


def measure_width_m(
    z,
    mask,
    fx,
    fy,
    cx,
    cy,
    band=(0.10, 0.28),
):
    v, u = np.where(
        mask
        & (z > 0)
        & np.isfinite(z)
    )

    if len(v) < 50:
        return None

    vt = v.min()
    vb = v.max()

    lo = vt + band[0] * (vb - vt)
    hi = vt + band[1] * (vb - vt)

    sel = (v >= lo) & (v <= hi)

    if sel.sum() < 20:
        return None

    X = (
        (u[sel] - cx)
        * z[v[sel], u[sel]]
        / fx
    )

    return float(
        np.percentile(X, 99)
        - np.percentile(X, 1)
    )


def unwrap_tensor(out):
    if isinstance(out, (list, tuple)):
        out = out[0]

    if isinstance(out, dict):
        out = next(iter(out.values()))

    if not torch.is_tensor(out):
        raise TypeError(
            f"모델 출력이 tensor가 아닙니다: {type(out)}"
        )

    return out


print("함수 OK")

In [ ]:
# 5. Sapiens Seg 1B 전신 마스크 + bbox
# RGBA alpha가 있으면 alpha 우선
# alpha가 없으면 Sapiens Seg 1B 사용

import os
import cv2
import numpy as np
import torch

SEG_REPO = "facebook/sapiens-seg-1b-torchscript"
SEG_FILE = (
    "sapiens_1b_goliath_best_goliath_mIoU_7994_"
    "epoch_151_torchscript.pt2"
)

SEG_URL = (
    f"https://huggingface.co/{SEG_REPO}/resolve/main/{SEG_FILE}"
)

SEG_PATH = "models/sapiens_seg_1b.pt2"

os.makedirs("models", exist_ok=True)
os.makedirs("work/mask", exist_ok=True)

ensure_model(
    SEG_URL,
    SEG_PATH,
    min_gb=3.5,
)

seg_model = torch.jit.load(SEG_PATH).cuda().eval()

MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32,
)

STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32,
)

HS = 1024
WS = 768


@torch.no_grad()
def sapiens_seg(rgb):
    """
    반환:
    label: 원본 RGB 크기의 class map
    mask : background 제외한 사람 전신 mask
    """

    H, W = rgb.shape[:2]

    x = cv2.resize(
        rgb,
        (WS, HS),
        interpolation=cv2.INTER_LINEAR,
    )

    x = x.astype(np.float32) / 255.0
    x = (x - MEAN) / STD

    x = torch.from_numpy(x)
    x = x.permute(2, 0, 1)
    x = x.unsqueeze(0).float().cuda()

    logits = unwrap_tensor(seg_model(x))

    if logits.ndim != 4:
        raise RuntimeError(
            f"Seg 출력 shape 오류: {tuple(logits.shape)}"
        )

    label = (
        logits.argmax(dim=1)[0]
        .detach()
        .cpu()
        .numpy()
        .astype(np.uint8)
    )

    label = cv2.resize(
        label,
        (W, H),
        interpolation=cv2.INTER_NEAREST,
    )

    # background=0, 사람 신체/의류/팔/다리/신발 등=1 이상
    mask = label > 0

    return label, mask


def keep_main_person(mask):
    """
    작은 오검출은 제거하고,
    화면 중앙에 가까운 가장 큰 사람 성분 하나만 유지.
    """

    H, W = mask.shape

    m = mask.astype(np.uint8)

    m = cv2.morphologyEx(
        m,
        cv2.MORPH_CLOSE,
        np.ones((7, 7), np.uint8),
        iterations=1,
    )

    count, labels, stats, centers = (
        cv2.connectedComponentsWithStats(
            m,
            connectivity=8,
        )
    )

    if count <= 1:
        return m.astype(bool)

    image_center = np.array(
        [W / 2, H / 2],
        dtype=np.float32,
    )

    best_id = None
    best_score = -np.inf

    for component_id in range(1, count):
        area = int(
            stats[
                component_id,
                cv2.CC_STAT_AREA,
            ]
        )

        # 전체 이미지의 0.2%보다 작은 성분은 제외
        if area < 0.002 * H * W:
            continue

        dist = np.linalg.norm(
            centers[component_id]
            - image_center
        )

        # 면적은 클수록 좋고, 중앙일수록 좋음
        score = area - 0.30 * (dist ** 2)

        if score > best_score:
            best_score = score
            best_id = component_id

    if best_id is None:
        best_id = (
            1
            + np.argmax(
                stats[
                    1:,
                    cv2.CC_STAT_AREA,
                ]
            )
        )

    out = (
        labels == best_id
    ).astype(np.uint8)

    out = cv2.morphologyEx(
        out,
        cv2.MORPH_CLOSE,
        np.ones((9, 9), np.uint8),
        iterations=1,
    )

    return out.astype(bool)


MASKS = {}

for f in FRAMES:
    rgb, alpha = load_rgb(f["rgb"])

    H, W = rgb.shape[:2]

    if alpha is not None:
        mask = keep_main_person(alpha)
        mask_source = "RGBA alpha"

    else:
        label, raw_mask = sapiens_seg(rgb)
        mask = keep_main_person(raw_mask)
        mask_source = "Sapiens Seg 1B"

        cv2.imwrite(
            f"work/mask/{f['key']}_seg_label.png",
            label,
        )

    if mask.sum() < 0.005 * H * W:
        raise RuntimeError(
            f"{f['key']}: 전신 마스크가 너무 작거나 비어 있음.\n"
            "work/mask 결과를 확인하세요."
        )

    ys, xs = np.where(mask)

    bbox = (
        int(xs.min()),
        int(ys.min()),
        int(xs.max()) + 1,
        int(ys.max()) + 1,
    )

    MASKS[f["key"]] = mask
    f["bbox"] = bbox
    f["hw"] = (H, W)

    overlay = rgb.copy()

    overlay[mask] = (
        0.50 * overlay[mask]
        + 0.50 * np.array([0, 180, 0])
    ).astype(np.uint8)

    x0, y0, x1, y1 = bbox

    cv2.rectangle(
        overlay,
        (x0, y0),
        (x1 - 1, y1 - 1),
        (255, 220, 0),
        3,
    )

    cv2.imwrite(
        f"work/mask/{f['key']}_maskvis.png",
        cv2.cvtColor(
            overlay,
            cv2.COLOR_RGB2BGR,
        ),
    )

    print(
        f"{f['key']}"
        f" | {mask_source}"
        f" | person={100 * mask.sum() / mask.size:.1f}%"
        f" | bbox={bbox}"
    )

del seg_model
torch.cuda.empty_cache()

print("\n✓ Seg 1B 완료")
print("work/mask/*_maskvis.png에서 전신이 잡혔는지 먼저 확인")

In [ ]:
# 6. crop
# STAND: margin 0.18
# TPOSE: margin 0.25

import os
import cv2
import numpy as np

os.makedirs("work/crop", exist_ok=True)

CROP = {}

for f in FRAMES:
    rgb, _ = load_rgb(f["rgb"])

    H, W = f["hw"]

    mask = MASKS[f["key"]]

    depth = (
        to_rgb_res(
            np.load(f["depth"]).astype(np.float32),
            W,
            H,
            nearest=True,
        )
        if f["depth"]
        else None
    )

    conf = (
        to_rgb_res(
            np.load(f["conf"]).astype(np.float32),
            W,
            H,
            nearest=True,
        )
        if f["conf"]
        else None
    )

    margin = (
        0.25
        if f["pose"] == "TPOSE"
        else 0.18
    )

    box = pad_bbox_3_4(
        *f["bbox"],
        margin=margin,
    )

    rgb_c, ib = crop_pad(
        rgb,
        box,
        fill=128,
    )

    mask_c = crop_pad(
        mask.astype(np.uint8),
        box,
        fill=0,
    )[0].astype(bool)

    depth_c = (
        crop_pad(
            depth,
            box,
            fill=0,
        )[0]
        if depth is not None
        else None
    )

    conf_c = (
        crop_pad(
            conf,
            box,
            fill=0,
        )[0]
        if conf is not None
        else None
    )

    CROP[f["key"]] = dict(
        rgb=rgb_c,
        mask=mask_c,
        depth=depth_c,
        conf=conf_c,
        ibox=ib,
    )

    cv2.imwrite(
        f"work/crop/{f['key']}_crop.png",
        cv2.cvtColor(
            rgb_c,
            cv2.COLOR_RGB2BGR,
        ),
    )

    ch, cw = rgb_c.shape[:2]

    print(
        f"{f['key']}"
        f" | crop={cw}x{ch}"
        f" | ratio={cw / ch:.3f}"
        f" | margin={margin:.2f}"
    )

print("\n✓ crop 완료")
print("work/crop에서 머리·발·TPOSE 양팔 끝이 안 잘렸는지 확인")

In [ ]:
# 7. Sapiens Depth 1B 추론

import os
import torch
import numpy as np
import cv2

DEPTH_REPO = "facebook/sapiens-depth-1b-torchscript"

DEPTH_FILE = (
    "sapiens_1b_render_people_epoch_88_"
    "torchscript.pt2"
)

DEPTH_URL = (
    f"https://huggingface.co/{DEPTH_REPO}/resolve/main/{DEPTH_FILE}"
)

DEPTH_PATH = "models/sapiens_depth_1b.pt2"

os.makedirs("models", exist_ok=True)

ensure_model(
    DEPTH_URL,
    DEPTH_PATH,
    min_gb=3.5,
)

dmodel = torch.jit.load(
    DEPTH_PATH,
).cuda().eval()


@torch.no_grad()
def sapiens_depth(rgb_c):
    ch, cw = rgb_c.shape[:2]

    x = cv2.resize(
        rgb_c,
        (WS, HS),
        interpolation=cv2.INTER_LINEAR,
    )

    x = x.astype(np.float32) / 255.0
    x = (x - MEAN) / STD

    x = torch.from_numpy(x)
    x = x.permute(2, 0, 1)
    x = x.unsqueeze(0).float().cuda()

    out = unwrap_tensor(dmodel(x))

    out = (
        out.squeeze()
        .float()
        .cpu()
        .numpy()
    )

    if out.ndim != 2:
        raise RuntimeError(
            f"Depth 출력 shape 오류: {out.shape}"
        )

    return cv2.resize(
        out,
        (cw, ch),
        interpolation=cv2.INTER_LINEAR,
    )


for f in FRAMES:
    CROP[f["key"]]["sap"] = sapiens_depth(
        CROP[f["key"]]["rgb"]
    )

    print(
        f"{f['key']}: "
        f"sap={CROP[f['key']]['sap'].shape}"
    )

del dmodel
torch.cuda.empty_cache()

print("\n✓ Depth 1B 완료")

In [ ]:
# 8. 사람 LiDAR anchor (+ 품질 경고: 점 개수 / inlier / 잔차)
import numpy as np, cv2, json, os
os.makedirs("work/out",exist_ok=True); REPORT=[]
MIN_ANCHOR=300   # 이보다 적으면 a,b 불안정 경고
for f in FRAMES:
    C=CROP[f['key']]; s=C['sap']; m=C['mask']
    if C['depth'] is None or C['conf'] is None:
        print(f"{f['key']}: depth/conf 없음 → skip"); continue
    m_er=cv2.erode(m.astype(np.uint8),np.ones((5,5),np.uint8),1).astype(bool)
    valid=m_er&(C['depth']>1e-6)&np.isfinite(C['depth'])&(C['conf']>=2)
    relaxed=False
    if valid.sum()<MIN_ANCHOR:   # conf==2 부족 → conf>=1 완화
        valid=m_er&(C['depth']>1e-6)&np.isfinite(C['depth'])&(C['conf']>=1); relaxed=True
    n_anc=int(valid.sum())
    if n_anc<50:
        print(f"{f['key']}: ⛔ anchor 점 {n_anc}개 — 너무 적음. 촬영 거리 가깝게/conf 점검. skip"); continue
    p=anchor_auto(s[valid].astype(np.float64), C['depth'][valid].astype(np.float64))
    z=apply_anchor(s,p); z[~m]=0.0; C['zmetric']=z; C['anchor']=p
    zf=z[m&(z>0)]
    warn=[]
    if n_anc<MIN_ANCHOR: warn.append(f'점부족({n_anc})')
    if relaxed: warn.append('conf>=1완화')
    if p['inlier']<0.6: warn.append(f"inlier낮음({p['inlier']*100:.0f}%)")
    if p['rel']>0.05: warn.append(f"잔차높음({p['rel']*100:.1f}%)")
    flag=('⚠ '+', '.join(warn)) if warn else '✓'
    REPORT.append(dict(key=f['key'],pose=f['pose'],view=f['view'],n_anchor=n_anc,relaxed=relaxed,
                       **{k:p[k] for k in ['mode','a','b','rel','inlier']},
                       z_med=float(np.median(zf)),z_min=float(zf.min()),z_max=float(zf.max()),flag=flag))
    print(f"{f['key']}: {p['mode']:9s} rel={p['rel']*100:.2f}% inlier={p['inlier']*100:.0f}% N={n_anc} z_med={np.median(zf):.3f}m  {flag}")
json.dump(REPORT,open("work/out/anchor_report.json","w"),indent=2,ensure_ascii=False)
print("\n경고(⚠) 있는 프레임은 a,b 신뢰 낮음 → 그 뷰 키가 튀면 여기 원인.")


In [ ]:
# 9. 선택: flying-pixel(tail) 제거 + Sapiens Normal 1B refine
# 순서: normal 추론 → tail 제거(erode+discontinuity+grazing) → Poisson refine
# 기본 ON. mask/crop/anchor/키 검증이 안정적일 때만.

DO_NORMAL_REFINE = True

# --- tail 제거 임계 (필요시 조정) ---
GRAZE_DEG = 75     # 작을수록 공격적(70~80). 시선과 이 각도 넘게 기운 면 제거
DISC_REL  = 0.03   # 이웃 depth 점프 임계(깊이 대비). 팔-몸통 과제거되면 0.05
ERODE_PX  = 2      # 실루엣 띠 제거 px

if DO_NORMAL_REFINE:
    import os
    import torch
    import numpy as np
    import cv2

    NORMAL_REPO = "facebook/sapiens-normal-1b-torchscript"
    NORMAL_FILE = (
        "sapiens_1b_normal_render_people_epoch_115_torchscript.pt2"
    )
    NORMAL_URL  = f"https://huggingface.co/{NORMAL_REPO}/resolve/main/{NORMAL_FILE}"
    NORMAL_PATH = "models/sapiens_normal_1b.pt2"

    ensure_model(NORMAL_URL, NORMAL_PATH, min_gb=3.5)
    nmodel = torch.jit.load(NORMAL_PATH).cuda().eval()

    @torch.no_grad()
    def sap_normal(rgb_c):
        ch, cw = rgb_c.shape[:2]
        x = cv2.resize(rgb_c, (WS, HS), interpolation=cv2.INTER_LINEAR)
        x = x.astype(np.float32) / 255.0
        x = (x - MEAN) / STD
        x = torch.from_numpy(x).permute(2, 0, 1).unsqueeze(0).float().cuda()
        n = unwrap_tensor(nmodel(x))
        if n.ndim == 4:
            n = n[0].permute(1, 2, 0)
        elif n.ndim == 3:
            n = n.permute(1, 2, 0)
        else:
            raise RuntimeError(f"Normal 출력 shape 오류: {tuple(n.shape)}")
        n = n.float().cpu().numpy()
        n = cv2.resize(n, (cw, ch), interpolation=cv2.INTER_LINEAR)
        return n / (np.linalg.norm(n, axis=-1, keepdims=True) + 1e-8)

    # ---------- tail(flying pixel) 제거 ----------
    def clean_flying(z, mask, fx, fy, cx, cy,
                     rel=DISC_REL, erode_px=ERODE_PX,
                     normal=None, graze_deg=GRAZE_DEG):
        M = mask & (z > 0)
        keep = M.copy()
        # (1) 실루엣 띠 제거 — tail의 뿌리
        if erode_px > 0:
            k = 2 * erode_px + 1
            keep &= cv2.erode(M.astype(np.uint8),
                              np.ones((k, k), np.uint8), 1).astype(bool)
        # (2) depth discontinuity — 날카로운 점프
        jump = np.zeros_like(z)
        for dy, dx in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
            sh = np.roll(np.roll(z, dy, 0), dx, 1)
            vd = np.roll(np.roll(M, dy, 0), dx, 1)
            jump = np.maximum(jump, np.where(vd, np.abs(z - sh), 0))
        keep &= jump < rel * np.clip(z, 1e-3, None)
        # (3) grazing — 시선과 크게 기운 면 (curtain tail 주범)
        if normal is not None:
            h, w = z.shape
            uu, vv = np.meshgrid(np.arange(w), np.arange(h))
            r = np.stack([(uu - cx) / fx, (vv - cy) / fy, np.ones_like(z)], -1)
            r /= np.linalg.norm(r, axis=-1, keepdims=True)
            keep &= np.abs((normal * r).sum(-1)) > np.cos(np.deg2rad(graze_deg))
        z2 = z.copy()
        z2[~keep] = 0.0
        return keep, z2

    # ---------- perspective screened-Poisson refine ----------
    def refine(z, n, mask, fx, fy, cx, cy, lam=0.1, iters=800):
        h, w = z.shape
        uu, vv = np.meshgrid(np.arange(w), np.arange(h))
        a = (uu - cx) / fx
        c = (vv - cy) / fy
        nx, ny, nz = n[..., 0], n[..., 1], n[..., 2]
        D = nx * a + ny * c + nz
        D = np.where(np.abs(D) < 1e-3, np.sign(D) * 1e-3 + 1e-9, D)
        Zu = -z * nx / (fx * D)
        Zv = -z * ny / (fy * D)
        reliable = np.abs(nz) > 0.3
        M = mask & (z > 0)
        er = cv2.erode(M.astype(np.uint8), np.ones((5, 5), np.uint8), 1).astype(bool)
        wd = np.where(M & ~er, lam * 6, lam)
        z0 = z.copy()
        d = z.copy()
        for _ in range(iters):
            num = np.zeros_like(d)
            den = np.zeros_like(d)
            lk = M[:, :-1] & M[:, 1:] & reliable[:, :-1]
            num[:, :-1] += np.where(lk, d[:, 1:] - Zu[:, :-1], 0); den[:, :-1] += lk
            num[:, 1:]  += np.where(lk, d[:, :-1] + Zu[:, :-1], 0); den[:, 1:]  += lk
            lv = M[:-1, :] & M[1:, :] & reliable[:-1, :]
            num[:-1, :] += np.where(lv, d[1:, :] - Zv[:-1, :], 0); den[:-1, :] += lv
            num[1:, :]  += np.where(lv, d[:-1, :] + Zv[:-1, :], 0); den[1:, :]  += lv
            d = np.where(M, (num + wd * z0) / (den + wd + 1e-9), z0)
        d[~mask] = 0.0
        return d

    for f in FRAMES:
        C = CROP[f["key"]]
        if "zmetric" not in C:
            continue

        H, W = f["hw"]
        x0, y0, _, _ = C["ibox"]
        fx, fy, cx, cy = get_intr(f["key"], W, H)
        cxc, cyc = cx - x0, cy - y0   # crop 기준 intrinsic

        nrm = sap_normal(C["rgb"])

        # tail 제거 → 깨끗한 mask/depth
        before = int((C["mask"] & (C["zmetric"] > 0)).sum())
        keep, z_clean = clean_flying(C["zmetric"], C["mask"], fx, fy, cxc, cyc, normal=nrm)
        after = int(keep.sum())

        # 깨끗해진 geometry 위에서 refine
        C["mask"]    = keep
        C["zmetric"] = refine(z_clean, nrm, keep, fx, fy, cxc, cyc)

        print(f"{f['key']}: tail 제거 {before}→{after} "
              f"({100*(before-after)/max(before,1):.1f}% 컷) + refine")

    del nmodel
    torch.cuda.empty_cache()
    print("✓ tail 제거 + Normal refine 완료")

else:
    print("normal refine OFF — tail 제거(grazing)도 같이 꺼짐. "
          "erode+discontinuity만 원하면 clean_flying(normal=None) 별도 사용")

In [ ]:
# 10. full-frame 복귀 + 키/너비 검증
# 중요:
# mf = zf > 0 이 아니라
# mf = MASKS[f['key']] 전신 Seg mask 사용

import numpy as np
import os
import json

os.makedirs("work/out", exist_ok=True)


def to_full(zc, ib, H, W):
    x0, y0, x1, y1 = ib

    full = np.zeros(
        (H, W),
        np.float32,
    )

    sx0 = max(0, x0)
    sy0 = max(0, y0)
    sx1 = min(W, x1)
    sy1 = min(H, y1)

    dx = sx0 - x0
    dy = sy0 - y0

    if sx1 > sx0 and sy1 > sy0:
        full[sy0:sy1, sx0:sx1] = zc[
            dy:dy + (sy1 - sy0),
            dx:dx + (sx1 - sx0),
        ]

    return full


HEIGHTS = {}
WIDTHS = {}

for f in FRAMES:
    C = CROP[f["key"]]

    if "zmetric" not in C:
        print(f"{f['key']}: metric depth 없음 → skip")
        continue

    H, W = f["hw"]

    zf = to_full(
        C["zmetric"],
        C["ibox"],
        H,
        W,
    )

    # 핵심 수정:
    # depth가 존재하는 픽셀을 사람으로 보지 않고
    # Sapiens Seg 1B 전신 mask 자체를 사람 기준으로 사용
    mf = MASKS[f["key"]]

    fx, fy, cx, cy = get_intr(
        f["key"],
        W,
        H,
    )

    h = measure_height_m(
        zf,
        mf,
        fx,
        fy,
        cx,
        cy,
    )

    wd = measure_width_m(
        zf,
        mf,
        fx,
        fy,
        cx,
        cy,
    )

    HEIGHTS.setdefault(
        f["pose"],
        {},
    )[f["view"]] = h

    WIDTHS.setdefault(
        f["pose"],
        {},
    )[f["view"]] = wd

    np.save(
        f"work/out/{f['key']}_person_metric.npy",
        zf,
    )

    if h is not None and wd is not None:
        print(
            f"{f['key']}: "
            f"키={h:.3f}m  "
            f"상단폭={wd:.3f}m"
        )

    else:
        print(f"{f['key']}: 측정 일부 실패")


def pair(d, a, b):
    if d.get(a) is not None and d.get(b) is not None:
        return (
            f"{a}={d[a]:.3f} "
            f"{b}={d[b]:.3f} "
            f"Δ={abs(d[a] - d[b]) * 100:.1f}cm"
        )

    return "측정값 부족"


print("\n=== pose별 4뷰 일관성 ===")

for pose in HEIGHTS:
    hv = HEIGHTS[pose]

    vals = [
        v
        for v in hv.values()
        if v is not None
    ]

    if vals:
        mean_h = np.mean(vals)
        std_h = np.std(vals)
        spread_h = max(vals) - min(vals)

        print(
            f"\n[{pose}] 키: "
            + "  ".join(
                f"{view}={value:.3f}"
                for view, value in hv.items()
                if value is not None
            )
        )

        print(
            f"mean={mean_h:.3f}m"
            f" | std={std_h * 100:.1f}cm"
            f" | spread={spread_h * 100:.1f}cm"
        )

        if SUBJECT_HEIGHT_M is not None:
            print(
                f"실제 키 {SUBJECT_HEIGHT_M:.3f}m 대비 "
                f"오차={(mean_h - SUBJECT_HEIGHT_M) * 100:+.1f}cm"
            )

    wv = WIDTHS.get(pose, {})

    print(
        "상단폭"
        f" | front/back: {pair(wv, 'front', 'back')}"
        f" | left/right: {pair(wv, 'left', 'right')}"
    )


json.dump(
    {
        "height": HEIGHTS,
        "width": WIDTHS,
    },
    open(
        "work/out/validate_report.json",
        "w",
    ),
    indent=2,
)

print("\n✓ validate_report.json 저장 완료")
print(
    "키 std가 약 2cm 이하이고, "
    "front/back 및 left/right 편차가 작으면 일관성 양호"
)

In [ ]:
# 10.5 실제 키 대비 절대오차 검증
# 일관성(std)은 이미 통과 → 여기선 "절대값이 맞나"만 본다.

MY_HEIGHT_M = 1.81   # ← 네 실제 키(맨발/신장계 기준)로 수정

import numpy as np

print(f"기준 실제 키: {MY_HEIGHT_M:.3f} m  (신발/머리카락 미포함 신장)\n")

all_vals = []
for pose in HEIGHTS:
    hv = HEIGHTS[pose]
    vals = [v for v in hv.values() if v is not None]
    if not vals:
        continue
    all_vals += vals
    mean_h = np.mean(vals)
    print(f"[{pose}]")
    for view, v in hv.items():
        if v is None:
            continue
        err = (v - MY_HEIGHT_M) * 100
        print(f"   {view:5s}: {v:.3f} m   오차 {err:+.1f} cm")
    print(f"   └ mean={mean_h:.3f} m  "
          f"평균오차 {(mean_h-MY_HEIGHT_M)*100:+.1f} cm  "
          f"std {np.std(vals)*100:.1f} cm\n")

g_mean = np.mean(all_vals)
bias = (g_mean - MY_HEIGHT_M) * 100
print("=" * 46)
print(f"전체 평균 키 : {g_mean:.3f} m")
print(f"전체 평균오차: {bias:+.1f} cm   (양수=실제보다 큼)")
print("=" * 46)

# 해석
if abs(bias) <= 2:
    print("✅ 충분. ±2cm 이내 — 신발/머리카락 실루엣 여유 범위. 그대로 진행.")
elif bias > 2:
    print(f"↑ {bias:+.1f}cm 큼. 거의 항상 정상 원인:")
    print("   - 측정이 '신발 밑창~머리카락 끝' 실루엣이라 신장보다 +2~5cm 큼")
    print("   - 머리 위 여유/신발창만큼 일관되게 큰 거면 OK (offset 빼면 됨)")
    print("   → bias가 4뷰 다 비슷하면 intrinsic 아니라 실루엣 정의 차이. 정상.")
else:
    print(f"↓ {bias:+.1f}cm 작음. 발/머리 일부 잘렸거나 intrinsic fy 미세 과대.")
    print("   → 마스크가 발끝/정수리 끊겼는지 work/mask vis 확인.")

# offset 보정값 (원하면 이후 키에서 빼서 신장으로)
print(f"\n참고: 신장 보정 offset = {bias:+.1f} cm "
      f"(측정 키에서 이만큼 빼면 실제 신장 근사)")

In [ ]:
# 11. zip 다운로드
import os
!zip -rq person_depth_out.zip work/out work/mask work/crop
print("zip:", round(os.path.getsize("person_depth_out.zip")/1e6,1), "MB")
from google.colab import files
files.download("person_depth_out.zip")
